# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

This notebook runs the **entire FastAPI backend + Qwen model** in Google Colab.

**Run cells in order. Do NOT skip any cell.**

---

## ⚠️ PHASE 1 — Clean Environment
Run this FIRST after every runtime restart.

In [ ]:
# CELL 1: Clean conflicting pre-installed packages
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Cleaned pre-installed packages")

## 📦 PHASE 2 — Install Dependencies (Pinned Versions)

In [ ]:
# CELL 2: Install core backend (PINNED — order matters)
!pip install fastapi==0.115.2 uvicorn==0.34.0 starlette==0.40.0 pydantic==2.12.0 pydantic-settings==2.12.0 httpx==0.28.1 python-multipart==0.0.18
print("\n✅ Core backend installed")

In [ ]:
# CELL 3: Install remaining dependencies
!pip install PyMuPDF==1.24.0 pdfplumber==0.11.0 Pillow==10.4.0 neo4j==5.23.0 pyngrok
print("\n✅ PDF parsing + Neo4j + ngrok installed")

In [ ]:
# CELL 4: Install AI/ML (uses Colab's pre-installed torch)
!pip install transformers accelerate
print("\n✅ Transformers + Accelerate installed")

## ✅ PHASE 3 — Validate Installation

In [ ]:
# CELL 5: Validate all imports work
import fastapi
import uvicorn
import pydantic
import starlette
import httpx
import fitz  # PyMuPDF
import pdfplumber
import neo4j
import torch
import transformers

print(f"fastapi     : {fastapi.__version__}")
print(f"uvicorn     : {uvicorn.__version__}")
print(f"pydantic    : {pydantic.__version__}")
print(f"starlette   : {starlette.__version__}")
print(f"httpx       : {httpx.__version__}")
print(f"PyMuPDF     : {fitz.version}")
print(f"neo4j       : {neo4j.__version__}")
print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"GPU         : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("\n✅ ALL IMPORTS VALID — No conflicts")

## 📂 PHASE 4 — Upload Backend Code

Upload your entire `backend/` folder from your local machine.

**Option A:** Use the cell below to clone from GitHub (if you pushed it).

**Option B:** Manually upload using the file browser on the left sidebar.

In [ ]:
# CELL 6: Create backend directory structure
import os

# Create all necessary directories
dirs = [
    '/content/backend/app',
    '/content/backend/app/routers',
    '/content/backend/app/services',
    '/content/backend/app/utils',
    '/content/backend/uploads',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Directory structure created at /content/backend/")
print("\nNow upload your backend files using ONE of these methods:")
print("  1. Drag & drop files into the left sidebar file browser")
print("  2. Use the next cell to upload via Google Drive")
print("  3. Use the cell after that to write files inline")

In [ ]:
# CELL 7 (OPTION A): Upload from Google Drive
# Uncomment these lines if your backend code is in Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/Knowledge_graph/backend/* /content/backend/
# print("✅ Backend files copied from Google Drive")

In [ ]:
# CELL 8 (OPTION B): Write all backend files inline
# This creates ALL the backend code directly in Colab.
# Run this cell if you did NOT upload files manually.

# ── __init__.py files ──
with open('/content/backend/app/__init__.py', 'w') as f:
    f.write('# Knowledge Graph Datasheet Backend\n')

with open('/content/backend/app/routers/__init__.py', 'w') as f:
    f.write('# Routers package\n')

with open('/content/backend/app/services/__init__.py', 'w') as f:
    f.write('# Services package\n')

# ── config.py ──
with open('/content/backend/app/config.py', 'w') as f:
    f.write('''import os\nfrom pydantic_settings import BaseSettings\n\nclass Settings(BaseSettings):\n    NEO4J_URI: str = "bolt://localhost:7687"\n    NEO4J_USER: str = "neo4j"\n    NEO4J_PASSWORD: str = "password"\n    QWEN_API_URL: str = ""\n    UPLOAD_DIR: str = "/content/backend/uploads"\n    CORS_ORIGINS: list[str] = ["*"]\n\n    class Config:\n        env_file = ".env"\n        env_file_encoding = "utf-8"\n\nsettings = Settings()\nos.makedirs(settings.UPLOAD_DIR, exist_ok=True)\n''')

print("✅ Config file created")
print("⚠️  You still need to upload the remaining files:")
print("    - app/models.py")
print("    - app/utils/__init__.py")
print("    - app/services/pdf_parser.py")
print("    - app/services/table_extractor.py")
print("    - app/services/content_detector.py")
print("    - app/services/graph_builder.py")
print("    - app/services/query_engine.py")
print("    - app/services/ai_client.py")
print("    - app/routers/upload.py")
print("    - app/routers/query.py")
print("    - app/main.py")
print("\nDrag & drop these from your local backend/app/ folder.")

## 🔗 PHASE 5 — Configure Neo4j Connection

You need a Neo4j instance accessible from Colab.

**Option A:** [Neo4j Aura Free](https://neo4j.com/cloud/aura-free/) — Free cloud Neo4j

**Option B:** Your local Neo4j (requires ngrok tunnel from your machine)

In [ ]:
# CELL 9: Set Neo4j credentials
# ⚠️ UPDATE THESE with your actual Neo4j connection details

import os

# For Neo4j Aura (cloud): use the connection URI from your Aura console
# For local Neo4j: use bolt://localhost:7687 (only works if Neo4j is on same machine)

os.environ['NEO4J_URI'] = 'bolt://localhost:7687'       # ← Change for Aura: neo4j+s://xxxxx.databases.neo4j.io
os.environ['NEO4J_USER'] = 'neo4j'                      # ← Your Neo4j username
os.environ['NEO4J_PASSWORD'] = 'password'                # ← Your Neo4j password
os.environ['QWEN_API_URL'] = ''                          # Leave empty — Qwen runs in same process

print(f"NEO4J_URI      = {os.environ['NEO4J_URI']}")
print(f"NEO4J_USER     = {os.environ['NEO4J_USER']}")
print(f"NEO4J_PASSWORD = {'*' * len(os.environ['NEO4J_PASSWORD'])}")
print("\n✅ Environment variables set")

## 🤖 PHASE 6 — Load Qwen Model

In [ ]:
# CELL 10: Load Qwen model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
print(f"GPU Available: {torch.cuda.is_available()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\n✅ Model loaded on: {model.device}")
print(f"   Model size: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B parameters")

In [ ]:
# CELL 11: Define Qwen generation function (used by the backend)
import json

SYSTEM_PROMPT = """You are a precise technical assistant for semiconductor datasheets.
You receive structured data extracted from a knowledge graph and must format it into
a clear, accurate natural language answer.

RULES:
1. ONLY use the data provided — DO NOT add any information.
2. If data is missing, say "not found in the datasheet".
3. Always include units when available.
4. Always include conditions/test conditions when available.
5. For tables, present data in a clean, readable format.
6. Never guess or approximate values.
"""

def generate_answer(query: str, data: dict, system_prompt: str = "") -> str:
    if not system_prompt:
        system_prompt = SYSTEM_PROMPT

    user_message = (
        f"User question: {query}\n\n"
        f"Knowledge Graph Data:\n{json.dumps(data, indent=2)}\n\n"
        f"Format a clear, precise answer using ONLY the data above."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
        )

    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Quick validation
test = generate_answer("What is the supply voltage?", {"parameter": "VCC", "min": "2.7", "max": "5.5", "unit": "V"})
print(f"✅ Qwen test: {test}")

## 🚀 PHASE 7 — Start FastAPI Server + ngrok Tunnel

In [ ]:
# CELL 12: Patch ai_client to use local Qwen instead of HTTP call
# This replaces the HTTP-based ai_client with a direct function call
# since the model is loaded in the same Colab runtime.

import sys
sys.path.insert(0, '/content/backend')

ai_client_code = '''
import json
import logging

logger = logging.getLogger(__name__)

# This will be set by the notebook after model loads
_generate_fn = None

def set_generate_function(fn):
    global _generate_fn
    _generate_fn = fn

async def format_with_qwen(query: str, structured_data: dict) -> str:
    if _generate_fn is None:
        logger.warning("Qwen model not loaded — returning raw data")
        return ""
    try:
        answer = _generate_fn(query, structured_data)
        return answer
    except Exception as e:
        logger.error("Qwen generation error: %s", e)
        return ""
'''

with open('/content/backend/app/services/ai_client.py', 'w') as f:
    f.write(ai_client_code)

print("✅ ai_client.py patched to use local Qwen model")

In [ ]:
# CELL 13: Wire up the Qwen model to the backend
from app.services.ai_client import set_generate_function
set_generate_function(generate_answer)
print("✅ Qwen model wired to backend ai_client")

In [ ]:
# CELL 14: Set up ngrok tunnel
# ⚠️ Get your FREE auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← REPLACE THIS!

from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Set auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open tunnel on port 8000
public_url = ngrok.connect(8000)

print("=" * 60)
print(f"🌐 YOUR PUBLIC BACKEND URL: {public_url}")
print("=" * 60)
print()
print("Use this URL in your frontend vite.config.js proxy target")
print("Or call it directly:")
print(f"  {public_url}/health")
print(f"  {public_url}/api/upload")
print(f"  {public_url}/api/query")

In [ ]:
# CELL 15: Start FastAPI server (THIS CELL BLOCKS — run it last!)
import uvicorn
import sys
sys.path.insert(0, '/content/backend')

print("🚀 Starting FastAPI server on port 8000...")
print("   Access via ngrok URL above")
print("   Press STOP (■) to shut down\n")

uvicorn.run("app.main:app", host="0.0.0.0", port=8000, log_level="info")